# 👷 Remoção de Pessoas com YOLOv8 Segmentation + Inpainting

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Detecção de Objetos e Inpainting

---

## Contexto

Fotografias de obra são tiradas durante o expediente — operários, técnicos e visitantes aparecem nas imagens de inspeção. Isso cria dois problemas práticos:

1. **Privacidade:** a LGPD exige consentimento para uso de imagens com rostos identificáveis. Armazenar fotos com pessoas em datasets de treinamento ou laudos digitais expõe a empresa a riscos legais.
2. **Qualidade do dado:** algoritmos de detecção de patologias (trincas, corrosão) treinados em imagens com pessoas aprendem artefatos espúrios — aprende que "trabalhador na frente de parede" é contexto de trinca, quando o que importa é a parede.

A solução é remover as pessoas automaticamente e **preencher a área removida de forma coerente com o fundo** (inpainting), de modo que a imagem resultante seja usável para inspeção e treinamento.

---

## Arquitetura do pipeline

Este notebook usa dois modelos clássicos de forma sequencial:

```
 Imagem colorida (BGR)
        │
        ▼
 ┌──────────────────────────────────────────┐
 │  YOLOv8-seg  (detecção + segmentação)   │
 │                                          │
 │  • Detecta bounding boxes de pessoas     │
 │  • Gera máscara poligonal por instância  │
 │  • Seleciona a MAIOR pessoa detectada    │
 └─────────────────┬────────────────────────┘
                   │  máscara binária
                   ▼
 ┌──────────────────────────────────────────┐
 │  Refinamento morfológico                 │
 │                                          │
 │  • MORPH_CLOSE  → fecha buracos          │
 │  • MORPH_ERODE  → reduz halo de borda    │
 └─────────────────┬────────────────────────┘
                   │  máscara refinada
                   ▼
 ┌──────────────────────────────────────────┐
 │  Inpainting em 2 estágios               │
 │                                          │
 │  Estágio 1 — INPAINT_NS (Navier-Stokes) │
 │    Propaga estruturas lineares (bordas,  │
 │    juntas, trincas) para dentro da área  │
 │                                          │
 │  Estágio 2 — INPAINT_TELEA              │
 │    Preenche com textura suave a partir   │
 │    da periferia para o centro            │
 └─────────────────┬────────────────────────┘
                   │
                   ▼
 ┌──────────────────────────────────────────┐
 │  Feathering (suavização de borda)        │
 │                                          │
 │  GaussianBlur na máscara → alpha blend   │
 │  Elimina borda abrupta original/inpaint  │
 └─────────────────┬────────────────────────┘
                   │
             IMAGEM FINAL
```

---

## YOLOv8: detecção vs. segmentação

A família YOLOv8 da Ultralytics oferece variantes para tarefas diferentes:

| Sufixo | Tarefa | Saída | Uso aqui |
|--------|--------|-------|----------|
| `yolov8n.pt` | Detecção | Bounding box (retângulo) | ❌ Grosseiro demais |
| **`yolov8n-seg.pt`** | **Segmentação** | **Máscara poligonal por instância** | ✅ |
| `yolov8n-pose.pt` | Pose estimation | Keypoints do corpo | ❌ Não gera máscara |

O sufixo `n` indica o modelo **nano** (o mais leve). Para melhor precisão em troca de mais memória e tempo: `s` → `m` → `l` → `x`.

## Inpainting: NS vs. Telea

| Algoritmo | Base matemática | Ponto forte | Limitação |
|-----------|-----------------|-------------|----------|
| **NS** (Navier-Stokes) | Equações de fluidos — propaga isófitas (linhas de mesmo brilho) | Preserva estruturas lineares (bordas, juntas) | Pode deixar área turva se a região for grande |
| **Telea** | Fast Marching Method — preenche da borda para o centro | Textura suave e natural | Pode borrar bordas nítidas |

Usar os dois em sequência (NS primeiro, Telea depois) aproveita os pontos fortes de cada um.

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Instalar dependências e testar com imagem sintética |
| 2 | Montar o Google Drive e configurar parâmetros |
| 3 | Definir o pipeline completo |
| 4 | Processar todas as imagens em lote |
| 5 | Visualizar resultados e analisar o relatório |
| 6 | Calibrar parâmetros com grade visual |

> 💡 **Recomendação:** execute as Seções 1 e 3 antes de montar o Drive. A Seção 1 valida o ambiente com uma imagem sintética — sem YOLO, sem fotos reais.

---
## Seção 1 — Instalar dependências e testar o ambiente

### 1a. Instalação

O único pacote extra necessário é `ultralytics`, que já inclui PyTorch e o download automático dos pesos do modelo na primeira execução.

In [ ]:
# Instala ultralytics (já inclui torch, torchvision, opencv)
try:
    from ultralytics import YOLO
    print('✅ ultralytics já instalado.')
except ImportError:
    print('📦 Instalando ultralytics...')
    import subprocess
    subprocess.run(['pip', 'install', 'ultralytics', '-q'], check=True)
    from ultralytics import YOLO
    print('✅ ultralytics instalado.')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

print(f'OpenCV  : {cv2.__version__}')
print(f'NumPy   : {np.__version__}')

### 1b. Teste com imagem sintética (sem YOLO, sem Drive)

Antes de usar o modelo real, validamos apenas as etapas de **inpainting e feathering** com uma máscara criada manualmente. Isso confirma que o pipeline de reconstrução da imagem funciona corretamente no seu ambiente.

**O que esperar:**
- A área mascarada (retângulo branco — simulando a pessoa) deve ser preenchida de forma coerente com o fundo
- A borda entre área preenchida e original deve ser suave (feathering)
- O comparativo NS × Telea deve mostrar diferença de textura

In [ ]:
# ── Criar imagem sintética: parede de concreto com textura + "pessoa" mascarada
h, w = 400, 600

# Fundo: concreto com gradiente + ruído
img_sint = np.zeros((h, w, 3), dtype=np.uint8)
for c, base in enumerate([140, 140, 135]):
    canal = np.full((h, w), base, dtype=np.float32)
    canal += np.random.normal(0, 12, (h, w))
    # Gradiente vertical suave
    canal += np.linspace(-15, 15, h)[:, None]
    img_sint[:, :, c] = np.clip(canal, 0, 255).astype(np.uint8)

# Juntas de concretagem (linhas horizontais)
for y in [120, 260]:
    cv2.line(img_sint, (0, y), (w, y), (100, 100, 98), 2)

# Trinca diagonal
for i in range(40, 200):
    j = int(350 + i * 0.5)
    if j < w:
        img_sint[i, j:j+2] = [50, 50, 50]

# Máscara sintética da "pessoa" (retângulo central)
mascara_sint = np.zeros((h, w), dtype=np.uint8)
cv2.rectangle(mascara_sint, (200, 60), (380, 360), 255, -1)

# ── Aplicar inpainting NS e Telea
etapa_ns    = cv2.inpaint(img_sint, mascara_sint, 5,  cv2.INPAINT_NS)
etapa_telea = cv2.inpaint(etapa_ns, mascara_sint, 9, cv2.INPAINT_TELEA)

# ── Feathering
feather = cv2.GaussianBlur(mascara_sint, (0, 0), 3)
alpha   = (feather.astype(np.float32) / 255.0)[..., None]
resultado_sint = (alpha * etapa_telea + (1 - alpha) * img_sint).astype(np.uint8)

# ── Visualizar
fig, eixos = plt.subplots(1, 5, figsize=(26, 5))
dados = [
    (img_sint,      'Original\n(parede + trinca)',    None),
    (mascara_sint,  'Máscara sintética\n("pessoa")',  'gray'),
    (etapa_ns,      'Após INPAINT_NS\n(Navier-Stokes)', None),
    (etapa_telea,   'Após INPAINT_TELEA\n(Fast Marching)', None),
    (resultado_sint,'Resultado final\n(+ feathering)', None),
]
for ax, (im, t, cmap) in zip(eixos, dados):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if cmap is None else im, cmap=cmap)
    ax.set_title(t, fontsize=9)
    ax.axis('off')

plt.suptitle('✅ Teste de ambiente — inpainting e feathering (sem YOLO)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Pipeline de inpainting OK.')
print('   Observe: a trinca diagonal fora da máscara deve ter sido preservada.')
print('   Pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar parâmetros

> ⚠️ **Pasta compartilhada?** Se `fotos_obra` não estiver em *Meu Drive*:  
> *Drive → clique direito → Organizar → Adicionar atalho ao Drive*

> 🔒 **LGPD:** os resultados serão salvos **localmente no Drive**, sem envio a serviços externos. O modelo YOLOv8 roda inteiramente no ambiente Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
# ── Caminhos
PASTA_ENTRADA = Path('/content/drive/MyDrive/fotos_obra')
PASTA_SAIDA   = Path('/content/drive/MyDrive/fotos_obra_sem_pessoas')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Modelo YOLO
# Opções por ordem de precisão crescente (e custo crescente):
#   yolov8n-seg.pt  → nano    (mais rápido, Colab grátis OK)
#   yolov8s-seg.pt  → small
#   yolov8m-seg.pt  → medium
#   yolov8l-seg.pt  → large   (requer GPU A100 ou Colab Pro)
MODELO = 'yolov8n-seg.pt'

# ── Parâmetros de detecção
CONFIANCA    = 0.4   # limiar de confiança (0.3 = mais detecções, 0.6 = mais conservador)

# ── Parâmetros de refinamento da máscara
ERODE_PX     = 3    # reduz halo na borda do corpo (2–5)
CLOSE_PX     = 7    # fecha buracos dentro da máscara (5–15)

# ── Parâmetros de inpainting
NS_RADIUS    = 5    # raio do NS — maior = mais propagação de estrutura
TELEA_RADIUS = 9    # raio do Telea — maior = área de busca de textura

# ── Feathering
FEATHER_SIGMA = 3   # sigma do blur na máscara (2–5)

# ── Redimensionamento
LARGURA_MAX  = 1200  # 0 = sem resize

# ── Extensões aceitas
EXTENSOES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# ── Verificar pasta
if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
else:
    arquivos = sorted([p for p in PASTA_ENTRADA.iterdir() if p.suffix.lower() in EXTENSOES])
    print(f'📂 Entrada     : {PASTA_ENTRADA}')
    print(f'📁 Saída       : {PASTA_SAIDA}')
    print(f'🤖 Modelo      : {MODELO}')
    print(f'⚙️  Confiança   : {CONFIANCA}')
    print(f'⚙️  Máscara     : close={CLOSE_PX}px, erode={ERODE_PX}px')
    print(f'⚙️  Inpainting  : NS radius={NS_RADIUS}, Telea radius={TELEA_RADIUS}')
    print(f'⚙️  Feathering  : sigma={FEATHER_SIGMA}')
    print(f'🖼️  {len(arquivos)} imagem(ns) encontrada(s)')

---
## Seção 3 — Definição do pipeline completo

Aqui definimos as três funções que compõem o pipeline. Execute esta célula uma vez — ela não processa nenhuma imagem ainda.

### Por que selecionar apenas a MAIOR pessoa?

Em fotos de obra, a pessoa de interesse para remoção geralmente é a que está mais próxima da câmera (logo, com maior área na imagem). Remover múltiplas pessoas simultaneamente aumenta a área de inpainting e piora o resultado quando as regiões se sobrepõem.

> Para remover **todas** as pessoas detectadas, altere `apenas_maior=True` para `False` na chamada da função.

In [ ]:
def detectar_mascara_pessoas(img_bgr, model, confianca=0.4, apenas_maior=True):
    """
    Roda YOLOv8-seg na imagem e retorna:
    - mascara_final : máscara binária (255=pessoa, 0=fundo)
    - n_detectadas  : número de pessoas detectadas
    """
    h, w = img_bgr.shape[:2]
    results = model(img_bgr, conf=confianca, verbose=False)

    mascaras  = []   # lista de (area, mascara)

    for r in results:
        if r.masks is None or r.boxes is None:
            continue

        cls_ids = r.boxes.cls.cpu().numpy().astype(int)
        tem_xy  = (getattr(r.masks, 'xy', None) is not None
                   and len(r.masks.xy) == len(cls_ids))

        for i, cid in enumerate(cls_ids):
            if cid != 0:   # classe 0 = person no COCO
                continue

            m = np.zeros((h, w), dtype=np.uint8)

            if tem_xy and r.masks.xy[i] is not None and len(r.masks.xy[i]) >= 3:
                pts = np.round(r.masks.xy[i]).astype(np.int32)
                cv2.fillPoly(m, [pts], 255)
            else:
                raw = r.masks.data[i].cpu().numpy()
                raw = cv2.resize(raw, (w, h), interpolation=cv2.INTER_NEAREST)
                m   = (raw > 0.5).astype(np.uint8) * 255

            area = int(m.sum() // 255)
            mascaras.append((area, m))

    if not mascaras:
        return None, 0

    if apenas_maior:
        _, mascara_final = max(mascaras, key=lambda x: x[0])
    else:
        # Une todas as máscaras
        mascara_final = np.zeros((h, w), dtype=np.uint8)
        for _, m in mascaras:
            mascara_final = cv2.bitwise_or(mascara_final, m)

    return mascara_final, len(mascaras)


def refinar_mascara(mascara, close_px=7, erode_px=3):
    """Fecha buracos e reduz halo de borda."""
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_px, close_px))
    mask    = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, k_close, iterations=1)
    if erode_px > 0:
        k_erode = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_px, erode_px))
        mask    = cv2.erode(mask, k_erode, iterations=1)
    return mask


def inpaint_com_feather(img_bgr, mascara, ns_radius=5, telea_radius=9, feather_sigma=3):
    """Inpainting em 2 estágios + feathering na borda."""
    etapa1    = cv2.inpaint(img_bgr, mascara, ns_radius,    cv2.INPAINT_NS)
    etapa2    = cv2.inpaint(etapa1,  mascara, telea_radius, cv2.INPAINT_TELEA)
    feather   = cv2.GaussianBlur(mascara, (0, 0), feather_sigma)
    alpha     = (feather.astype(np.float32) / 255.0)[..., None]
    resultado = (alpha * etapa2 + (1 - alpha) * img_bgr).astype(np.uint8)
    return resultado


def processar_imagem(caminho, model,
                     confianca=0.4, close_px=7, erode_px=3,
                     ns_radius=5, telea_radius=9, feather_sigma=3,
                     largura_max=1200, apenas_maior=True):
    """
    Função principal: recebe caminho, devolve
    (img_orig, mascara_refinada, resultado, stats_dict)
    """
    img = cv2.imread(str(caminho))
    assert img is not None, f'Não foi possível abrir: {caminho.name}'

    if largura_max > 0 and img.shape[1] > largura_max:
        r   = largura_max / img.shape[1]
        img = cv2.resize(img, (largura_max, int(img.shape[0] * r)),
                         interpolation=cv2.INTER_AREA)

    mascara_bruta, n_detectadas = detectar_mascara_pessoas(img, model, confianca, apenas_maior)

    stats = {
        'arquivo'           : caminho.name,
        'largura'           : img.shape[1],
        'altura'            : img.shape[0],
        'pessoas_detectadas': n_detectadas,
        'pessoa_removida'   : False,
        'area_mascara_pct'  : 0.0,
    }

    if mascara_bruta is None:
        return img, None, img, stats

    mascara = refinar_mascara(mascara_bruta, close_px, erode_px)
    resultado = inpaint_com_feather(img, mascara, ns_radius, telea_radius, feather_sigma)

    stats['pessoa_removida']  = True
    stats['area_mascara_pct'] = round(100.0 * int(mascara.sum() // 255) / mascara.size, 2)

    return img, mascara, resultado, stats


print('✅ Funções do pipeline definidas. Prossiga para a Seção 4.')

---
## Seção 4 — Processamento em lote

### O que é salvo por imagem

```
fotos_obra_sem_pessoas/
  └── nome_foto/
        ├── nome_foto_original.png      ← imagem de entrada (redimensionada se aplicável)
        ├── nome_foto_mascara.png       ← máscara binária da pessoa detectada
        ├── nome_foto_resultado.png     ← imagem com pessoa removida
        └── nome_foto_comparativo.png  ← original | máscara overlay | resultado
  └── relatorio_remocao.csv            ← tabela consolidada com métricas
```

> ⏱️ **Tempo estimado:** ~3–8 s por imagem com `yolov8n-seg` na GPU do Colab grátis. O NLMeans do módulo anterior é mais lento; YOLO com GPU é rápido.

In [ ]:
# ── Carregar modelo (faz download automático na primeira vez)
print(f'🔄 Carregando {MODELO}...')
model = YOLO(MODELO)
print(f'✅ Modelo carregado.')

In [ ]:
todas_stats = []
erros       = []

for arq in tqdm(arquivos, desc='Removendo pessoas'):
    try:
        img_orig, mascara, resultado, stats = processar_imagem(
            arq, model,
            confianca    = CONFIANCA,
            close_px     = CLOSE_PX,
            erode_px     = ERODE_PX,
            ns_radius    = NS_RADIUS,
            telea_radius = TELEA_RADIUS,
            feather_sigma= FEATHER_SIGMA,
            largura_max  = LARGURA_MAX,
        )

        # Pasta individual por imagem
        pasta_img = PASTA_SAIDA / arq.stem
        pasta_img.mkdir(parents=True, exist_ok=True)

        cv2.imwrite(str(pasta_img / f'{arq.stem}_original.png'), img_orig)
        cv2.imwrite(str(pasta_img / f'{arq.stem}_resultado.png'), resultado)

        if mascara is not None:
            cv2.imwrite(str(pasta_img / f'{arq.stem}_mascara.png'), mascara)

            # Overlay vermelho para o comparativo
            overlay = img_orig.copy()
            px = img_orig[mascara > 0]
            cor = np.full_like(px, [0, 0, 200])
            overlay[mascara > 0] = cv2.addWeighted(px, 0.4, cor, 0.6, 0)

            comp = np.hstack([img_orig, overlay, resultado])
            # Legenda no comparativo
            cv2.rectangle(comp, (10, 10), (620, 58), (255, 255, 255), -1)
            cv2.putText(comp,
                        f'Original | Mascara (overlay) | Resultado   Pessoas: {stats["pessoas_detectadas"]}',
                        (18, 44), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (30, 30, 30), 2, cv2.LINE_AA)
            cv2.imwrite(str(pasta_img / f'{arq.stem}_comparativo.png'), comp)
        else:
            comp = np.hstack([img_orig, resultado])
            cv2.imwrite(str(pasta_img / f'{arq.stem}_comparativo.png'), comp)

        todas_stats.append(stats)

    except Exception as e:
        erros.append((arq.name, str(e)))

# ── Salvar relatório CSV
df_stats = pd.DataFrame(todas_stats)
caminho_relatorio = PASTA_SAIDA / 'relatorio_remocao.csv'
df_stats.to_csv(str(caminho_relatorio), index=False)

total_removidas = df_stats['pessoa_removida'].sum()
print(f'\n✅ {len(todas_stats)} imagem(ns) processada(s)')
print(f'   Pessoas removidas  : {total_removidas}')
print(f'   Sem pessoa dectada : {len(todas_stats) - total_removidas}')
print(f'   Relatório salvo em : {caminho_relatorio}')

if erros:
    print('\n⚠️  Erros:')
    for nome, msg in erros:
        print(f'   {nome}: {msg}')

---
## Seção 5 — Visualização de resultados e análise do relatório

### 5a. Painel de quatro vistas

| Vista | O que observar |
|-------|----------------|
| **Original** | Referência — pessoa visível |
| **Máscara refinada** | Deve cobrir o corpo inteiro. Se estiver incompleta, aumente `CLOSE_PX`. Se capturar fundo ao redor, aumente `ERODE_PX` ou reduza `CONFIANCA`. |
| **Resultado** | A área preenchida deve ter textura coerente com o entorno. Halo visível → aumente `FEATHER_SIGMA`. Área muito borrada → reduza `TELEA_RADIUS`. |
| **Mapa de diferença** | Branco = onde a imagem mudou. Deve coincidir com a máscara — qualquer alteração fora dela indica bug. |

In [ ]:
MAX_EXIBIR = 4

for arq in arquivos[:MAX_EXIBIR]:
    img_orig, mascara, resultado, stats = processar_imagem(
        arq, model,
        confianca=CONFIANCA, close_px=CLOSE_PX, erode_px=ERODE_PX,
        ns_radius=NS_RADIUS, telea_radius=TELEA_RADIUS,
        feather_sigma=FEATHER_SIGMA, largura_max=LARGURA_MAX,
    )

    diff = cv2.absdiff(img_orig, resultado)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    diff_norm = cv2.normalize(diff_gray, None, 0, 255, cv2.NORM_MINMAX)

    fig, eixos = plt.subplots(1, 4, figsize=(22, 5))
    vistas = [
        (img_orig,   'Original',                  None),
        (mascara if mascara is not None else np.zeros_like(diff_norm),
                     'Máscara refinada',           'gray'),
        (resultado,  'Resultado (sem pessoa)',      None),
        (diff_norm,  'Mapa de diferença\n(branco = alterado)', 'hot'),
    ]
    for ax, (im, t, cmap) in zip(eixos, vistas):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if cmap is None else im, cmap=cmap)
        ax.set_title(t, fontsize=9)
        ax.axis('off')

    det = stats['pessoas_detectadas']
    rem = '✅ removida' if stats['pessoa_removida'] else '⚠️ não detectada'
    plt.suptitle(f'{arq.name} — {det} pessoa(s) detectada(s) — {rem}',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 5b. Relatório consolidado

Tabela com métricas de todas as imagens processadas. A coluna `area_mascara_pct` indica quanto da imagem foi preenchida por inpainting — valores acima de 30% geralmente resultam em qualidade inferior (a área é grande demais para reconstrução coerente).

In [ ]:
df_exib = df_stats.copy()
df_exib.index = range(1, len(df_exib) + 1)

def colorir_linha(row):
    if not row['pessoa_removida']:
        return ['background-color: #f0f0f0'] * len(row)   # cinza = sem pessoa
    if row['area_mascara_pct'] > 30:
        return ['background-color: #ffcccc'] * len(row)   # vermelho = área grande
    if row['area_mascara_pct'] > 15:
        return ['background-color: #fff3cd'] * len(row)   # amarelo = atenção
    return ['background-color: #d4edda'] * len(row)       # verde = OK

display(df_exib.style.apply(colorir_linha, axis=1))

print('\n🟢 Verde   : remoção bem sucedida (área < 15%)')
print('🟡 Amarelo : área removida entre 15–30% — verificar visualmente')
print('🔴 Vermelho: área removida > 30% — inpainting pode ser inconsistente')
print('⬜ Cinza   : nenhuma pessoa detectada')

---
## Seção 6 — Calibração de parâmetros

### Guia de diagnóstico

| Problema observado | O que ajustar |
|--------------------|---------------|
| Pessoa não detectada | Reduzir `CONFIANCA` (ex: 0.3) |
| Máscara com buracos (roupa vs. pele) | Aumentar `CLOSE_PX` (ex: 11–15) |
| Halo visível ao redor da pessoa removida | Aumentar `ERODE_PX` (ex: 5–7) |
| Borda abrupta entre inpaint e original | Aumentar `FEATHER_SIGMA` (ex: 5–7) |
| Área preenchida muito borrada | Reduzir `TELEA_RADIUS` (ex: 5–7) |
| Estruturas (juntas, linhas) apagadas dentro da área | Aumentar `NS_RADIUS` (ex: 7–9) |

A grade abaixo varia `CONFIANCA` e `FEATHER_SIGMA` para a primeira imagem.

In [ ]:
if arquivos:
    img_ref = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img_ref.shape[1] > LARGURA_MAX:
        r       = LARGURA_MAX / img_ref.shape[1]
        img_ref = cv2.resize(img_ref, (LARGURA_MAX, int(img_ref.shape[0] * r)),
                             interpolation=cv2.INTER_AREA)

    confs    = [0.3, 0.4, 0.5]
    sigmas   = [2, 3, 5]

    fig, eixos = plt.subplots(len(confs), len(sigmas),
                               figsize=(6 * len(sigmas), 5 * len(confs)))

    for i, conf in enumerate(confs):
        for j, sigma in enumerate(sigmas):
            mask_g, n_det = detectar_mascara_pessoas(img_ref, model, conf)
            if mask_g is not None:
                mask_g  = refinar_mascara(mask_g, CLOSE_PX, ERODE_PX)
                res_g   = inpaint_com_feather(img_ref, mask_g,
                                              NS_RADIUS, TELEA_RADIUS, sigma)
            else:
                res_g = img_ref

            eixos[i][j].imshow(cv2.cvtColor(res_g, cv2.COLOR_BGR2RGB))
            eixos[i][j].set_title(
                f'conf={conf}  feather_sigma={sigma}\n'
                f'{n_det} pessoa(s) detectada(s)', fontsize=8)
            eixos[i][j].axis('off')

    plt.suptitle(f'Grade de parâmetros — {arquivos[0].name}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('👆 Escolha a combinação com melhor resultado e atualize na Seção 2.')

---
## Próximos passos

Com as imagens limpas em `fotos_obra_sem_pessoas`, o pipeline de preparação do dataset fica completo:

```
fotos_obra/
    │
    ├─ [módulo 03] CLAHE              → fotos_obra_clahe/
    ├─ [módulo 07] Remoção artefatos  → fotos_obra_clean/
    ├─ [este módulo] Remoção pessoas  → fotos_obra_sem_pessoas/
    └─ [módulo flip] Augmentation     → fotos_obra_flip/
                          │
                          ▼
                    Dataset final para treino
                    U-Net / YOLOv8 / SAM
```

### Limitações conhecidas do inpainting clássico

| Situação | Comportamento esperado |
|----------|------------------------|
| Pessoa na frente de fundo uniforme (parede lisa) | ✅ Excelente resultado |
| Pessoa parcialmente fora do frame | ✅ Bom resultado |
| Pessoa na frente de padrão regular (azulejos, grade) | ⚠️ Padrão pode não se alinhar |
| Grupo de pessoas se sobrepondo | ❌ Preferir segmentação individual ou modelo de difusão |
| Sombra da pessoa no fundo | ⚠️ Sombra não é detectada pelo YOLO — permanece |

Para casos complexos, considere modelos de inpainting baseados em difusão (LaMa, Stable Diffusion Inpaint) que produzem resultados visualmente mais coerentes em grandes áreas.

### Referências

- Redmon, J. & Farhadi, A. (2018). *YOLOv3: An Incremental Improvement*. arXiv:1804.02767
- Jocher, G. et al. (2023). *Ultralytics YOLOv8*. https://github.com/ultralytics/ultralytics
- Bertalmío, M. et al. (2001). *Navier-Stokes, Fluid Dynamics, and Image and Video Inpainting*. CVPR.
- Telea, A. (2004). *An Image Inpainting Technique Based on the Fast Marching Method*. Journal of Graphics Tools, 9(1).
- LGPD — Lei nº 13.709/2018, Art. 5º, II (dado pessoal — imagem como dado biométrico)
